In [1]:
import numpy as np
from scipy.special import erfc

def Green(Fo, d, k, a, t, n, icr):
    #------------------------------------------------------------------------
    #  Purpose:
    #     compute Green's function for solid layers
    #
    #  Variable Description:
    #     Fo    - Fourier number
    #     d     - thickness
    #     k,a   - thermal conductivity and diffusivity
    #     t     - time
    #------------------------------------------------------------------------
#t= tl, a= alR or alW or alG, k = kR or kG or kW, d = dR or dG or dW

    Fo_cr = 1 / np.pi / np.sqrt(2)  # characteristic Fourier number
    dt = t[1] - t[0]                # time step, in s
    t_cr = 300 * icr                # critical nondimensional time (in s)
    x = np.array([0, d])

    if t[-1] < t_cr:
        nt = len(t)
    else:
        nt = int(np.ceil(t_cr / dt))

    g = np.zeros((len(t), 2))

    I1 = np.where((Fo[:nt] <= Fo_cr) & (Fo[:nt] != 0))[0]  # indices for small time solution
    I2 = np.where((Fo[:nt] > Fo_cr) & (Fo[:nt] != 0))[0]   # indices for large time solution

    if len(I1) > 0:
        # Compute small time solution
        R = np.arange(-int(np.floor((n - 1) / 2)), int(np.ceil((n - 1) / 2)) + 1)
        xx, tt, nn = np.meshgrid(x, t[I1], R, indexing='ij')
        K = (np.sqrt(a * tt / np.pi) * np.exp(-(xx - 2 * nn * d) ** 2 / (4 * a * tt)) -
             np.abs(xx - 2 * nn * d) / 2 * erfc(np.abs(xx - 2 * nn * d) / (2 * np.sqrt(a * tt))))
        g[I1, :] = 2 / k * np.sum(K, axis=2)

    if len(I2) > 0:
        # Solution based on eigenfunction expansion
        R = np.arange(1, n + 1)
        xx, tt, nn = np.meshgrid(x, t[I2], R, indexing='ij')
        K = (np.exp(-a * (nn * np.pi / d) ** 2 * tt) / nn ** 2 * np.cos(nn * np.pi * xx / d))
        xx, tt = np.meshgrid(x, t[I2], indexing='ij')
        g[I2, :] = (a * tt / k / d +
                    d / (6 * k) * (3 * (1 - xx / d) ** 2 - 1) -
                    2 * d / np.pi ** 2 / k * np.sum(K, axis=2))

    return g
